# 13 · Data Manipulation (INSERT / UPDATE / DELETE)

DML changes the **data** inside tables.
- `INSERT` (single & multiple rows, insert-from-select)
- `UPDATE` (always mind the `WHERE`!)
- `DELETE`
- `UPSERT` (`INSERT ... ON CONFLICT`)

> Again we work on a throwaway `demo_inventory` table, rebuilt at the top so you
> can re-run freely and never disturb the course data.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## Set up a sandbox table

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_inventory;
CREATE TABLE demo_inventory (
    sku       TEXT PRIMARY KEY,
    name      TEXT NOT NULL,
    quantity  INTEGER NOT NULL DEFAULT 0,
    price     REAL NOT NULL
);
SELECT 'ready' AS status;

## `INSERT` multiple rows

In [ ]:
%%sql
INSERT INTO demo_inventory (sku, name, quantity, price) VALUES
    ('A1', 'Widget',  100, 2.50),
    ('A2', 'Gadget',   40, 9.99),
    ('A3', 'Gizmo',     0, 14.00);
SELECT * FROM demo_inventory;

## `UPDATE`
Raise all prices by 10%. The `WHERE` here targets one row; **omit `WHERE` and every row changes** — a common, costly mistake.

In [ ]:
%%sql
UPDATE demo_inventory SET price = ROUND(price * 1.10, 2) WHERE sku = 'A2';
SELECT * FROM demo_inventory;

## `UPDATE` many rows
Restock everything that's out of stock:

In [ ]:
%%sql
UPDATE demo_inventory SET quantity = 25 WHERE quantity = 0;
SELECT * FROM demo_inventory;

## `DELETE`
Remove cheap items. Again, mind the `WHERE`.

In [ ]:
%%sql
DELETE FROM demo_inventory WHERE price < 3;
SELECT * FROM demo_inventory;

## `UPSERT` — insert or update on conflict
Insert a row; if the primary key already exists, update it instead. Run this
cell twice and watch the quantity change rather than erroring.

In [ ]:
%%sql
INSERT INTO demo_inventory (sku, name, quantity, price)
VALUES ('A2', 'Gadget', 5, 9.99)
ON CONFLICT(sku) DO UPDATE SET quantity = quantity + excluded.quantity;
SELECT * FROM demo_inventory WHERE sku = 'A2';

## `INSERT ... SELECT`
Copy selected course products into our sandbox:

In [ ]:
%%sql
INSERT INTO demo_inventory (sku, name, quantity, price)
SELECT 'P' || product_id, product_name, in_stock, unit_price
FROM products
WHERE category_id = 2;               -- Books
SELECT * FROM demo_inventory ORDER BY sku;

## Clean up

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_inventory;
SELECT 'cleaned up' AS status;

## Practice

**✏️ Exercise 1.** Create a table demo_tasks(id integer pk, title text, done integer default 0). Insert two tasks, then mark the first as done (done = 1). Select all.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_tasks;
CREATE TABLE demo_tasks (id INTEGER PRIMARY KEY, title TEXT, done INTEGER DEFAULT 0);
INSERT INTO demo_tasks (title) VALUES ('Learn INSERT'), ('Learn UPDATE');
UPDATE demo_tasks SET done = 1 WHERE id = 1;
SELECT * FROM demo_tasks;

### ✅ Recap
`INSERT` adds rows (including from a `SELECT`), `UPDATE` changes them, `DELETE`
removes them — and `WHERE` is what keeps those last two from hitting every row.
`UPSERT` merges inserts with updates.

**Next:** `14_views_and_indexes.ipynb`.